# M06 — Lazy, plan y cache (teoría de clase)

Sin acción no hay job. `cache()` no materializa hasta un `count`/`show`.


## Arranque

Ejecuta esta celda y la siguiente. Kernel: **Python (NovaShop)**.


In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
spark = get_spark('novashop-clase-m06')
print(spark.version, spark.sparkContext.master)


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col

base = spark.createDataFrame([Row(x=i, canal="web" if i % 2 == 0 else "app") for i in range(20)])
planned = base.where(col("x") > 3).where(col("canal") == "web").select("x")
print("sin acción:", planned)
print("con count:", planned.count())
planned.explain("formatted")


In [ ]:
warm = planned.cache()
print("1º (materializa)", warm.count())
print("2º (debería leer cache)", warm.count())
warm.unpersist()


Mira Spark UI (4040): el primer `count` llena Storage; el segundo no relee el plan desde cero.

**Siguiente:** [M06-01](../../labs/M06-optimizacion-ejecucion/M06-01-explain-dag.md).
